# Unir tablas con pandas

Este notebook muestra como hacer joins entre las tablas de `data/` usando `pandas.merge()`.

Vamos a ver tres casos:

1. Unir una tabla de hechos con una tabla maestra.
2. Unir tres tablas cuando hay una tabla puente.
3. Usar `left join` para no perder registros.


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

data_path = Path("data")

empleados = pd.read_csv(data_path / "empleados.csv")
empleado_turno = pd.read_csv(data_path / "empleado_turno.csv")
turnos = pd.read_csv(data_path / "turnos.csv")
inventario = pd.read_csv(data_path / "inventario.csv")
materiales = pd.read_csv(data_path / "materiales.csv")
proveedores = pd.read_csv(data_path / "proveedores.csv")

print("Tablas cargadas correctamente")


In [ ]:
for nombre, tabla in {
    "empleados": empleados,
    "empleado_turno": empleado_turno,
    "turnos": turnos,
    "inventario": inventario,
    "materiales": materiales,
    "proveedores": proveedores,
}.items():
    print(f"\n{nombre}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas")
    print(tabla.columns.tolist())


## Regla basica para unir tablas

La estructura general es:

```python
tabla_resultado = tabla_izquierda.merge(
    tabla_derecha,
    on="columna_clave",
    how="inner"
)
```

- `on` indica la columna que tienen en comun.
- `how="inner"` deja solo coincidencias.
- `how="left"` conserva todo lo de la tabla izquierda.

Antes de unir, siempre conviene revisar:

- Cual es la llave primaria de cada tabla.
- Si el nombre de la clave es igual en ambas tablas.
- Si la relacion es `1:1`, `1:m` o `m:m`.


## Ejemplo 1: inventario + materiales + proveedores

Queremos saber el stock por material, con el nombre del material y el proveedor.

In [ ]:
inventario_detalle = (
    inventario
    .merge(materiales, on="id_material", how="left", validate="m:1")
    .merge(proveedores, on="id_proveedor", how="left", validate="m:1", suffixes=("_material", "_proveedor"))
)

inventario_detalle[[
    "fecha",
    "id_material",
    "nombre_material",
    "tipo_material",
    "cantidad",
    "nombre_proveedor",
    "pais"
]].head(10)


Fijate en la logica:

- `inventario` tiene muchas filas por material.
- `materiales` tiene una fila por material.
- `proveedores` tiene una fila por proveedor.

Por eso usamos `validate="m:1"`: muchas filas de la izquierda contra una fila de la derecha.

## Ejemplo 2: empleados + tabla puente + turnos

Aqui la union no se hace en un solo paso porque `empleados` y `turnos` se conectan por la tabla puente `empleado_turno`.

In [ ]:
turnos["fecha"] = pd.to_datetime(turnos["fecha"])

empleados_turnos = (
    empleado_turno
    .merge(empleados, on="id_empleado", how="left", validate="m:1")
    .merge(turnos, on="id_turno", how="left", validate="m:1")
)

empleados_turnos[[
    "id_turno",
    "fecha",
    "tipo_turno",
    "id_empleado",
    "nombre",
    "rol",
    "horas_trabajadas",
    "puntualidad"
]].head(12)


In [ ]:
resumen_empleado = (
    empleados_turnos
    .groupby(["id_empleado", "nombre", "departamento"], as_index=False)
    .agg(
        turnos_asignados=("id_turno", "count"),
        horas_totales=("horas_trabajadas", "sum")
    )
    .sort_values(["horas_totales", "turnos_asignados"], ascending=False)
)

resumen_empleado.head(10)


Ese patron es muy comun en bases relacionales:

- tabla maestra: `empleados`
- tabla puente: `empleado_turno`
- otra tabla maestra: `turnos`

Cuando haya una tabla puente, normalmente el merge se hace en cadena.

## Ejemplo 3: left join para conservar todos los empleados

Si queremos conservar todos los empleados, incluso los que no tengan registros en otra tabla, usamos `how="left"`.

In [ ]:
empleados_con_turnos = empleados.merge(
    empleado_turno,
    on="id_empleado",
    how="left",
    indicator=True
)

empleados_con_turnos[["id_empleado", "nombre", "rol", "id_turno", "_merge"]].head(15)


In [ ]:
conteo_coincidencias = empleados_con_turnos["_merge"].value_counts(dropna=False)
conteo_coincidencias


## Mini guia para ellos

Antes de hacer cualquier join, hagan estas preguntas:

1. Cual es la tabla principal que quiero analizar.
2. Cual es la clave para conectarla con otra tabla.
3. Necesito solo coincidencias (`inner`) o conservar todo (`left`).
4. Hay tabla puente o la union es directa.

Si quieren practicar mas, pueden intentar:

- Unir `dosimetria` con `empleados`.
- Unir `mantenimientos` con `reactores`.
- Unir `empleado_capacitacion` con `empleados` y `capacitaciones`.
